# Pulizia del dataset (BRFSS 2015)

Il dataset è il questionario sanitario americano **BRFSS 2015**: 328.135 persone × 321 feature, target `_MICHD` (infarto / malattia coronarica, ~8.8% positivi).
Il 45% delle celle è NaN, ma i NaN **non sono casuali**: dipendono dalla struttura del questionario. Abbiamo individuato 4 cause:

| causa | esempio | colonne | trattamento |
|---|---|---|---|
| **1. Fisso vs cellulare**: due versioni del questionario | `CTELENUM` vs `CTELNUM1` | ~18 | eliminate (info sulla chiamata, non sulla salute) |
| **2. Intervista interrotta** (`DISPCODE`=1200, 15% delle righe) | `INCOME2`, `EXERANY2`, `_BMI5` | ~40 | NaN veri → imputazione (passo successivo) |
| **3. Domande condizionali**: chieste solo se la risposta precedente è "sì" | `AVEDRNK2` solo a chi beve | ~45 | il NaN significa "non applicabile" → 0 dove è una quantità |
| **4. Moduli opzionali** scelti dai singoli stati | modulo asma, vista, diabete | ~100 | quasi tutte eliminate o soglia sui NaN (passo successivo) |

Inoltre il BRFSS codifica le non-risposte con **codici numerici** (7/77/777 = "non so", 9/99/999 = "rifiuta") e lo zero con codici come 88 = "nessun giorno": vanno convertiti, altrimenti un modello lineare li legge come valori enormi.

In questo notebook:
1. eliminiamo le colonne inutili (identificativi, gestione della chiamata, pesi statistici, versione fisso/cellulare);
2. mettiamo **0 dove la domanda non è stata posta perché la risposta precedente era "no"** (zeri strutturali);
3. convertiamo i codici "nessuno" (88, 888, 555) in 0;
4. convertiamo i codici "non so / rifiuta" in NaN.

Le funzioni sono in `cleaning.py` (solo numpy).

## 0. Caricamento dei dati (con cache)

Leggere i CSV con `load_csv_data` richiede 1–2 minuti. La prima volta li leggiamo e salviamo le matrici in formato `.npy` nella cartella `cache/`; dalle volte successive le carichiamo da lì in pochi secondi.
Se cambiano i CSV basta cancellare la cartella `cache/`.
I nomi delle colonne si leggono a parte dall'intestazione di `x_train.csv`, perché `load_csv_data` restituisce solo i numeri e toglie la colonna `Id`.

In [49]:
import numpy as np
import sys
from pathlib import Path

# --- 1. Trova la cartella del progetto (quella che contiene data/ e cleaning.py) ---
def find_project_dir(start):
    for p in [start, *start.parents]:
        if (p / "data").is_dir() and (p / "cleaning.py").is_file():
            return p
    return None

project_dir = find_project_dir(Path.cwd())
assert project_dir is not None, "non trovo la cartella del progetto (deve contenere data/ e cleaning.py)"
if str(project_dir) not in sys.path:
    sys.path.append(str(project_dir))

data_path  = project_dir / "data"      # ml-project1/data
cache_path = project_dir / "cache"     # ml-project1/cache
cache_path.mkdir(exist_ok=True)

print("cartella progetto :", project_dir)
print("cartella data esiste?", data_path.exists())
print("cartella cache    :", cache_path)

# --- 2. Caricamento dati: dalla cache .npy se esiste, altrimenti dai CSV ---
if (cache_path / "x_train.npy").exists():
    x_train = np.load(cache_path / "x_train.npy")
    x_test  = np.load(cache_path / "x_test.npy")
    y_train = np.load(cache_path / "y_train.npy")
    train_ids = np.load(cache_path / "train_ids.npy")
    test_ids  = np.load(cache_path / "test_ids.npy")
    print("--> Caricati dalla cache")
else:
    from helpers import load_csv_data
    x_train, x_test, y_train, train_ids, test_ids = load_csv_data(str(data_path))
    np.save(cache_path / "x_train.npy", x_train)
    np.save(cache_path / "x_test.npy", x_test)
    np.save(cache_path / "y_train.npy", y_train)
    np.save(cache_path / "train_ids.npy", train_ids)
    np.save(cache_path / "test_ids.npy", test_ids)
    print("--> Letti dai CSV e salvati in cache")

# --- 3. Nomi delle colonne (load_csv_data toglie la colonna Id) ---
from cleaning import (read_header, fill_structural_zeros, recode_none_to_zero, find_special_codes,
                      special_codes_to_nan, informativita_colonne, identify_columns_to_drop, drop_columns, colonne_ridondanti)

cols = read_header(data_path / "x_train.csv")
if len(cols) == x_train.shape[1] + 1:
    cols = cols[1:]
assert len(cols) == x_train.shape[1], "le intestazioni non combaciano con la matrice"

# --- 4. Controlli finali ---
print("train:", x_train.shape, " test:", x_test.shape)
print("colonne (feature):", len(cols))
print("positivi (target = 1): %d su %d  = %.2f%%" % ((y_train == 1).sum(), len(y_train), 100 * (y_train == 1).mean()))
print("NaN iniziali: %.1f%%" % (100 * np.isnan(x_train).mean()))

cartella progetto : /Users/giovannipavan/Desktop/EPFL/Corsi/Primo Semestre/ML/projects/ml-project1
cartella data esiste? True
cartella cache    : /Users/giovannipavan/Desktop/EPFL/Corsi/Primo Semestre/ML/projects/ml-project1/cache
--> Caricati dalla cache
train: (328135, 321)  test: (109379, 321)
colonne (feature): 321
positivi (target = 1): 28975 su 328135  = 8.83%
NaN iniziali: 44.8%


## 1. Colonne da eliminare a priori

- **Identificativi e date**: non contengono informazione sulla persona.
- **Gestione della chiamata** e **pesi statistici del sondaggio**: descrivono il campionamento, non la salute.
- **Gruppo 1, versione cellulare del questionario**: le colonne che esistono solo per chi è stato chiamato al cellulare (le corrispondenti del fisso sono già nella prima lista). Il loro NaN indica solo il tipo di telefono.
- **`HAREHAB1`** (riabilitazione dopo un infarto): è compilata solo per chi ha già avuto un infarto (target = 1 nel 100% dei 773 casi) → **leakage**.
- **Colonne grezze ridondanti**: ne esiste una versione già pulita, calcolata dal CDC:
  - `WEIGHT2`, `HEIGHT3` mescolano libbre/pollici e kg/cm → si usano `WTKG3`, `HTM4`, `_BMI5`;
  - `EXEROFT1/2`, `EXERHMM1/2` mescolano "volte a settimana" (1xx) e "volte al mese" (2xx) → si usano `PAFREQ*_`, `PADUR*_`, `PAMIN*_`;
  - `FLSHTMY2`, `HIVTSTD3` sono date (mese/anno).

In [50]:
da_togliere = [
    # Identificativi e date
    'Id', '_STATE', 'FMONTH', 'IDATE', 'IMONTH', 'IDAY', 'IYEAR', 'STATERES',
    # Amministrative (gestione della chiamata)
    'DISPCODE', 'SEQNO', '_PSU', 'CTELENUM', 'PVTRESD1', 'CELLFON3',
    'NUMHHOL2', 'CPDEMO1', 'QSTVER', 'QSTLANG', 'MSCODE',
    # Pesi statistici del sondaggio
    '_STSTR', '_STRWT', '_RAWRAKE', '_WT2RAKE', '_DUALUSE', '_DUALCOR', '_LLCPWT',
]

gruppo1_telefono = [
    # versione fisso: domande sulla famiglia / casa (solo chi è chiamato al fisso)
    'COLGHOUS', 'LADULT', 'NUMADULT', 'NUMMEN', 'NUMWOMEN', 'NUMPHON2',
    # versione cellulare
    'CTELNUM1', 'CELLFON2', 'CADULT', 'PVTRESD2', 'CCLGHOUS', 'CSTATE',
    'LANDLINE', 'HHADULT',
]

leakage = ['HAREHAB1']

ridondanti = ['WEIGHT2', 'HEIGHT3',                          # -> WTKG3, HTM4, _BMI5
              'EXEROFT1', 'EXERHMM1', 'EXEROFT2', 'EXERHMM2',  # -> PAFREQ*_, PADUR*_
              'FLSHTMY2', 'HIVTSTD3']                          # date

def drop_columns(X, cols, names):
    """Restituisce (X senza le colonne 'names', lista dei nomi rimanenti)."""
    keep = [j for j, c in enumerate(cols) if c not in set(names)]
    return X[:, keep], [cols[j] for j in keep]

print("gruppo 'identificativi/amministrative/pesi':", len(da_togliere))
print("gruppo 'fisso vs cellulare' (NaN = tipo di telefono):", len(gruppo1_telefono))
print("leakage (HAREHAB1):", len(leakage), "-> compilata solo da chi ha avuto un infarto")
print("colonne grezze ridondanti:", len(ridondanti))

tutte = da_togliere + gruppo1_telefono + leakage + ridondanti
x_train, cols_tr = drop_columns(x_train, cols, tutte)
x_test,  cols_te = drop_columns(x_test,  cols, tutte)
assert cols_tr == cols_te, "train e test hanno colonne diverse"
cols = cols_tr
print("--> colonne rimaste:", len(cols), "(erano 321)")
print("NaN dopo il drop: %.1f%%" % (100 * np.isnan(x_train).mean()))

gruppo 'identificativi/amministrative/pesi': 26
gruppo 'fisso vs cellulare' (NaN = tipo di telefono): 14
leakage (HAREHAB1): 1 -> compilata solo da chi ha avuto un infarto
colonne grezze ridondanti: 8
--> colonne rimaste: 273 (erano 321)
NaN dopo il drop: 46.5%


## 2. Zeri strutturali (domande condizionali)

Alcune domande sono poste solo a chi ha risposto "sì" a una domanda precedente (*madre*). Chi ha risposto "no" ha NaN nella domanda *figlia*, ma la risposta vera la conosciamo: **0**.
Esempio: `AVEDRNK2` ("quanti drink bevi in media?") è chiesta solo a chi ha bevuto nell'ultimo mese (`ALCDAY5` ≠ 888).

| persona | `ALCDAY5` (madre) | `AVEDRNK2` prima | dopo |
|---|---|---|---|
| A | 205 = ha bevuto 5 giorni | 2 | 2 |
| B | 888 = nessun giorno | NaN | **0** |
| C | NaN = intervista interrotta | NaN | NaN (resta un NaN vero) |

Mettiamo 0 **solo dove la madre dice esplicitamente "no"**:

| colonne figlie | madre | 0 se la madre vale |
|---|---|---|
| `AVEDRNK2`, `DRNK3GE5`, `MAXDRNKS` (alcol) | `ALCDAY5` | 888 = nessun giorno |
| `POORHLTH` (giorni limitati dalla salute) | `PHYSHLTH`, `MENTHLTH` | entrambe 88 = nessun giorno |
| attività fisica principale (`PAFREQ1_`, `PADUR1_`, `PAMIN11_`, `PA1MIN_`, `PAVIG11_`, `PA1VIGM_`, `_MINAC11`, `METVL11_`, `ACTIN11_`) | `EXERANY2` | 2 = non fa attività |
| seconda attività (`PAFREQ2_`, `PADUR2_`, `PAMIN21_`, `PAVIG21_`, `_MINAC21`, `METVL21_`, `ACTIN21_`) | `EXERANY2` / `EXRACT21` | 2 / 88 = nessuna seconda attività |

Caso speciale: `DIABAGE2` (età alla diagnosi del diabete) viene trasformata in **anni con il diabete** = `_AGE80` − `DIABAGE2`, con 0 per i non diabetici. Mettere 0 direttamente nell'età alla diagnosi non avrebbe senso. Tra i diabetici il tasso di infarto sale dal 17% (≤ 4 anni di diabete) al 29% (> 18 anni).

Le colonne madri restano nel dataset: così il modello distingue lo "0 vero" dallo "0 perché la domanda non è stata posta".

In [51]:
def nan_frac(X, cols, names):
    ix = {c: i for i, c in enumerate(cols)}
    return {n: round(float(np.isnan(X[:, ix[n]]).mean()), 3) for n in names if n in ix}

check = ['AVEDRNK2', 'MAXDRNKS', 'POORHLTH', 'PA1MIN_', 'PAMIN21_', 'DIABAGE2']
prima = nan_frac(x_train, cols, check)
print("NaN prima degli zeri strutturali:")
for c in check:
    print(f"   {c:9s} {prima[c]:6.1%}")

x_train = fill_structural_zeros(x_train, cols)
x_test  = fill_structural_zeros(x_test,  cols)

dopo = nan_frac(x_train, cols, check)
print("NaN dopo gli zeri strutturali (0 solo dove la madre dice 'no'):")
for c in check:
    print(f"   {c:9s} {prima[c]:6.1%}  ->  {dopo[c]:6.1%}   (delta {dopo[c]-prima[c]:+.1%})")
print("NaN totali train: %.1f%%" % (100 * np.isnan(x_train).mean()))

NaN prima degli zeri strutturali:
   AVEDRNK2   52.1%
   MAXDRNKS   52.3%
   POORHLTH   48.7%
   PA1MIN_    34.4%
   PAMIN21_   36.3%
   DIABAGE2   87.1%
NaN dopo gli zeri strutturali (0 solo dove la madre dice 'no'):
   AVEDRNK2   52.1%  ->    5.0%   (delta -47.1%)
   MAXDRNKS   52.3%  ->    5.2%   (delta -47.1%)
   POORHLTH   48.7%  ->    0.0%   (delta -48.7%)
   PA1MIN_    34.4%  ->   10.1%   (delta -24.3%)
   PAMIN21_   36.3%  ->   11.8%   (delta -24.5%)
   DIABAGE2   87.1%  ->    1.0%   (delta -86.1%)
NaN totali train: 43.9%


## 3. Codici che significano "nessuno" → 0

Alcune colonne non usano lo 0 ma un codice: `88` = "nessun giorno / nessuno", `888` = "mai", `555` = "mai" (frutta e verdura).
Lasciati così, il modello leggerebbe per esempio "88 giorni di malessere" in un mese che ne ha 30.

- `PHYSHLTH`, `MENTHLTH`, `POORHLTH`, `CHILDREN`, `DRNK3GE5`: 88 → 0
- `ALCDAY5`, `STRENGTH`: 888 → 0
- `FRUITJU1`, `FRUIT1`, `FVBEANS`, `FVGREEN`, `FVORANG`, `VEGETAB1`: 555 → 0

Questo passo va fatto **dopo** il passo 2 (che legge i codici 88/888 delle madri) e **prima** del passo 4.

In [52]:
from cleaning import NONE_CODES

# Contiamo quante celle cambiano, prima di applicare la ricodifica
conta = {}
for col, codes in NONE_CODES.items():
    if col in cols:
        j = cols.index(col)
        conta[col] = int(np.isin(x_train[:, j], codes).sum())

x_train = recode_none_to_zero(x_train, cols)
x_test  = recode_none_to_zero(x_test,  cols)

print("Codici 'nessuno / mai' -> 0 (contati sul train):")
for col, n in conta.items():
    print(f"   {col:10s} {NONE_CODES[col]} -> 0   su {n:7d} celle")
print("totale celle ricodificate:", sum(conta.values()))
print("NaN totali train: %.1f%%" % (100 * np.isnan(x_train).mean()))

Codici 'nessuno / mai' -> 0 (contati sul train):
   PHYSHLTH   [88] -> 0   su  204264 celle
   MENTHLTH   [88] -> 0   su  224183 celle
   POORHLTH   [88] -> 0   su   93548 celle
   CHILDREN   [88] -> 0   su  240617 celle
   DRNK3GE5   [88] -> 0   su  117010 celle
   ALCDAY5    [888] -> 0   su  154556 celle
   STRENGTH   [888] -> 0   su  181546 celle
   FRUITJU1   [555] -> 0   su  123451 celle
   FRUIT1     [555] -> 0   su   12035 celle
   FVBEANS    [555] -> 0   su   50956 celle
   FVGREEN    [555] -> 0   su   23287 celle
   FVORANG    [555] -> 0   su   44094 celle
   VEGETAB1   [555] -> 0   su    5509 celle
totale celle ricodificate: 1475056
NaN totali train: 43.9%


## 4. Codici "non so / rifiuta" → NaN

Il BRFSS codifica le non-risposte con 7/77/777/7777 ("non so") e 9/99/999/9999 ("rifiuta"). La lunghezza del codice dipende da quante cifre ha la risposta.
Regola applicata a ogni colonna non continua, guardando il suo valore massimo:
- se il massimo è 9, 99, 999, … → i valori 9…9 e 7…7 diventano NaN;
- con una sola cifra il 7 è un codice solo se il 6 non compare: in `EMPLOY1` il 7 è "pensionato", una risposta valida;
- le colonne continue (BMI, minuti di attività, anni con diabete, …) vengono saltate;
- alcuni codici fuori schema sono indicati a mano (es. `_AGEG5YR` = 14, `_DRNKWEK` = 99900).

I codici vengono individuati **sul train** e applicati identici al test, così le due matrici sono trasformate nello stesso modo.\n`verbose=True` stampa, per ogni colonna, quali codici sono stati convertiti e su quante righe.

In [53]:
# I codici si individuano sul TRAIN e si applicano identici al TEST
codes = find_special_codes(x_train, cols)

n_col = 0
n_celle = 0
for nome, codici in codes.items():
    j = cols.index(nome)
    n = int(np.isin(x_train[:, j], codici).sum())
    if n:
        n_col += 1
        n_celle += n

x_train = special_codes_to_nan(x_train, cols, codes, verbose=True)
x_test  = special_codes_to_nan(x_test,  cols, codes)

print("--> riepilogo: %d colonne con codici speciali, %d celle trasformate in NaN (train)" % (n_col, n_celle))
print("NaN totali train: %.1f%%   test: %.1f%%" % (100 * np.isnan(x_train).mean(), 100 * np.isnan(x_test).mean()))

GENHLTH    codici [7, 9] -> NaN su 886 righe
PHYSHLTH   codici [77, 99] -> NaN su 7107 righe
MENTHLTH   codici [77, 99] -> NaN su 5359 righe
POORHLTH   codici [77, 99] -> NaN su 3557 righe
HLTHPLN1   codici [7, 9] -> NaN su 1326 righe
PERSDOC2   codici [7, 9] -> NaN su 1333 righe
MEDCOST    codici [7, 9] -> NaN su 830 righe
CHECKUP1   codici [7, 9] -> NaN su 4445 righe
BPHIGH4    codici [7, 9] -> NaN su 917 righe
BPMEDS     codici [7, 9] -> NaN su 216 righe
BLOODCHO   codici [7, 9] -> NaN su 6940 righe
CHOLCHK    codici [7, 9] -> NaN su 4305 righe
TOLDHI2    codici [7, 9] -> NaN su 2523 righe
CVDSTRK3   codici [7, 9] -> NaN su 737 righe
ASTHMA3    codici [7, 9] -> NaN su 984 righe
ASTHNOW    codici [7, 9] -> NaN su 1350 righe
CHCSCNCR   codici [7, 9] -> NaN su 815 righe
CHCOCNCR   codici [7, 9] -> NaN su 654 righe
CHCCOPD1   codici [7, 9] -> NaN su 1547 righe
HAVARTH3   codici [7, 9] -> NaN su 1882 righe
ADDEPEV2   codici [7, 9] -> NaN su 1450 righe
CHCKIDNY   codici [7, 9] -> NaN su 9

## Risultato

Dopo la pulizia i NaN rimasti sono solo **NaN veri**: interviste interrotte, "non so / rifiuta" e moduli opzionali non somministrati.
Il passo successivo è decidere, per questi:
- **colonne dei moduli opzionali** (> 80–90% NaN): eliminarle, tenendo eventualmente quelle utili come `INSULIN` (modulo diabete) con le categorie "non diabetico / sconosciuto / sì / no";
- **colonne rimaste**: imputare con la mediana (continue) o la moda (categoriche), calcolate solo sul train, più un indicatore di missing per le colonne in cui il NaN è frequente (es. `INCOME2`, 17%).

In [54]:
nan_col = np.isnan(x_train).mean(axis=0)
print("colonne: %d   NaN totali: %.1f%%" % (len(cols), 100 * np.isnan(x_train).mean()))
print("colonne con NaN sopra una certa soglia:")
for t in [0.0, 0.1, 0.5, 0.6, 0.8, 0.9]:
    print(f"   > {t:4.0%} NaN: {(nan_col > t).sum():4d}")

# Colonne appena sopra / appena sotto la soglia del 60%: da tenere d'occhio
print("colonne vicine alla soglia 60% (decisive per il taglio):")
for j in np.argsort(-nan_col):
    if 0.55 <= nan_col[j] <= 0.70:
        segno = "TENUTA " if nan_col[j] <= 0.6 else "tolta  "
        print(f"   {segno} {cols[j]:12s} {100*nan_col[j]:5.1f}% NaN")

colonne: 273   NaN totali: 45.6%
colonne con NaN sopra una certa soglia:
   >   0% NaN:  262
   >  10% NaN:  159
   >  50% NaN:  126
   >  60% NaN:  123
   >  80% NaN:  111
   >  90% NaN:   94
colonne vicine alla soglia 60% (decisive per il taglio):
   tolta   _PNEUMO2      69.7% NaN
   tolta   LMTJOIN3      69.5% NaN
   tolta   ARTHSOCL      69.5% NaN
   tolta   _FLSHOT6      68.7% NaN
   tolta   SXORIENT      63.2% NaN
   tolta   TRNSGNDR      62.7% NaN
   TENUTA  BPMEDS        59.9% NaN
   TENUTA  SMOKDAY2      58.4% NaN
   TENUTA  IMFVPLAC      56.8% NaN


## Proviamo a togliere altre colonne

Fin qui abbiamo tolto 48 colonne "a priori" (identificativi, pesi, fisso/cellulare, leakage, grezze ridondanti) e abbiamo ripulito i valori.
Restano **273 colonne** con il **45,6% di NaN**.

`informativita_colonne` ci dice, per ogni colonna:

- `rate_nan` / `rate_val`: tasso di infarto fra chi ha NaN e fra chi ha risposto → **il NaN è informativo?**
- `spread`: quanto cambia il tasso di infarto fra i valori della colonna (0 = la colonna non distingue nulla)
- `pos_nan` / `pos_val`: quanti infarti in ciascun gruppo (per capire se il segnale è rumore)

Poi `identify_columns_to_drop` trova le colonne con più NaN di una soglia.

In [55]:
print("--- informativita' di ogni colonna (ordinate per max(info_nan, spread)) ---")
res = informativita_colonne(x_train, y_train, cols)
print("colonne analizzate:", len(res))

# le 5 piu' informative: spread alto E quota di risposta alta
nan_col = np.isnan(x_train).mean(axis=0)
quota = {r[0]: 1 - nan_col[cols.index(r[0])] for r in res}
peso = sorted(((r[5] * quota[r[0]], r[0], r[5], quota[r[0]]) for r in res), reverse=True)
print("--- 'peso' = spread x quota di risposta (le prime 10) ---")
print(f"{'colonna':<12}{'spread':>8}{'quota':>9}{'peso':>8}")
for w, nome, sp, q in peso[:10]:
    print(f"{nome:<12}{sp:8.2f}{100*q:8.1f}%{w:8.3f}")

--- informativita' di ogni colonna (ordinate per max(info_nan, spread)) ---
colonna        %NaN  rate_nan  rate_val  info_nan  spread  pos_nan  pos_val
STREHAB1       99.8       8.8      35.7      26.9     1.8    28791      184
CVDSTRK3        0.2      31.5       8.8      22.7     2.3      232    28743
CDHELP         99.1       8.6      28.8      20.1     0.7    28095      880
CHCCOPD1        0.5      23.5       8.8      14.7     2.9      363    28612
CDASSIST       97.1       8.4      22.7      14.2     4.0    26815     2160
CDDISCUS       97.1       8.4      22.6      14.2     2.6    26818     2157
CDSOCIAL       97.1       8.4      22.6      14.1     3.1    26844     2131
CDHOUSE        97.1       8.4      22.6      14.1     4.0    26846     2129
CHCKIDNY        0.3      22.9       8.8      14.1     1.4      224    28751
BLDSUGAR       93.5       7.9      22.0      14.1     2.5    24283     4692
EYEEXAM        93.5       7.9      22.0      14.0     0.9    24284     4691
INSULIN     

In [56]:
drop_nan = identify_columns_to_drop(x_train, cols, 0.6)
print("colonne con > 60% di NaN:", len(drop_nan))
print("quota di NaN di queste colonne: da %.1f%% a %.1f%%" % (
    100 * np.isnan(x_train[:, [cols.index(c) for c in drop_nan]]).mean(axis=0).min(),
    100 * np.isnan(x_train[:, [cols.index(c) for c in drop_nan]]).mean(axis=0).max()))
print("colonne senza nessun NaN:", int((np.isnan(x_train).mean(axis=0) == 0).sum()))
print(drop_nan)

colonne con > 60% di NaN: 123
quota di NaN di queste colonne: da 62.7% a 99.9%
colonne senza nessun NaN: 11
['ASTHNOW', 'PREGNANT', 'STOPSMK2', 'LASTSMK2', 'LMTJOIN3', 'ARTHDIS2', 'ARTHSOCL', 'JOINPAIN', 'WHRTST10', 'PDIABTST', 'PREDIAB1', 'INSULIN', 'BLDSUGAR', 'FEETCHK2', 'DOCTDIAB', 'CHKHEMO3', 'FEETCHK', 'EYEEXAM', 'DIABEYE', 'DIABEDU', 'CAREGIV1', 'CRGVREL1', 'CRGVLNG1', 'CRGVHRS1', 'CRGVPRB1', 'CRGVPERS', 'CRGVHOUS', 'CRGVMST2', 'CRGVEXPT', 'VIDFCLT2', 'VIREDIF3', 'VIPRFVS2', 'VINOCRE2', 'VIEYEXM2', 'VIINSUR2', 'VICTRCT4', 'VIGLUMA2', 'VIMACDG2', 'CIMEMLOS', 'CDHOUSE', 'CDASSIST', 'CDHELP', 'CDSOCIAL', 'CDDISCUS', 'WTCHSALT', 'LONGWTCH', 'DRADVISE', 'ASTHMAGE', 'ASATTACK', 'ASERVIST', 'ASDRVIST', 'ASRCHKUP', 'ASACTLIM', 'ASYMPTOM', 'ASNOSLEP', 'ASTHMED3', 'ASINHALR', 'STREHAB1', 'CVDASPRN', 'ASPUNSAF', 'RLIVPAIN', 'RDUCHART', 'RDUCSTRK', 'ARTTODAY', 'ARTHWGT', 'ARTHEXER', 'ARTHEDU', 'TETANUS', 'HPVADVC2', 'HPVADSHT', 'SHINGLE2', 'HADMAM', 'HOWLONG', 'HADPAP2', 'LASTPAP2', 'HPVTES

In [44]:
import importlib
import cleaning
importlib.reload(cleaning)

# A questo punto puoi rifare l'import
from cleaning import informativita_colonne
print("cleaning.py ricaricato (le modifiche sono attive)")

cleaning.py ricaricato (le modifiche sono attive)


In [57]:
SOGLIA_NAN = 0.6
CANDIDATI  = ["INSULIN", "CVDASPRN", "DRADVISE"]   # tenuti da parte per la CV

drop_nan = identify_columns_to_drop(x_train, cols, SOGLIA_NAN)      # 123
print("colonne con > %.0f%% NaN: %d" % (100 * SOGLIA_NAN, len(drop_nan)))

candidati = [c for c in CANDIDATI if c in drop_nan]
drop_finale = [c for c in drop_nan if c not in candidati]
print("tenuti da parte (non li buttiamo, li testiamo in CV):", candidati)
print("da togliere davvero:", len(drop_finale))
print("colonne rimaste dopo il taglio: %d  (273 - %d)" % (len(cols) - len(drop_finale), len(drop_finale)))

colonne con > 60% NaN: 123
tenuti da parte (non li buttiamo, li testiamo in CV): ['INSULIN', 'CVDASPRN', 'DRADVISE']
da togliere davvero: 120
colonne rimaste dopo il taglio: 153  (273 - 120)


### Perché tenere `INSULIN`, `CVDASPRN` e `DRADVISE`

Sono le **3 sole colonne** che escludiamo dal taglio automatico del 60% di NaN. Non le teniamo "per principio": le teniamo perché sono le uniche 3 del gruppo con una **ragione specifica**, e le mettiamo da parte per un test in cross-validation.

| colonna | %NaN | spread | quota di risposta | tasso infarti se "sì" | se "no" |
|---|---|---|---|---|---|
| `INSULIN` | 93,4% | 3,98 | 6,6% | **27,98%** | 18,96% |
| `CVDASPRN` | 95,7% | 6,86 | 4,3% | **20,84%** | 4,48% |
| `DRADVISE` | 90,7% | 6,03 | 9,3% | **20,72%** | 6,50% |
| *(riferimento)* `GENHLTH` | 0,3% | 5,37 | 99,7% | — | — |

**`INSULIN`** — l'unica con una ragione clinica chiara: *tra i diabetici, chi usa insulina ha il 28% di infarti contro il 19% di chi non la usa*. Il suo NaN però **mescola due cose diverse** ("non sono diabetico" e "sono diabetico ma il mio stato non ha somministrato il modulo"), quindi da solo è ambiguo: serve `DIABETE3` per separare i due casi (vedi la cella dopo).

**`CVDASPRN` e `DRADVISE`** (aspirina) — hanno gli spread più alti del gruppo, ma coprono solo il 4% e il 9% della popolazione. Sono state pensate per un sospetto di *leakage*, perché l'aspirina viene prescritta **dopo** un infarto: più che una causa, sarebbero una conseguenza. Il controllo dice che il sospetto è **solo parzialmente fondato**:

- **non è leakage strutturale**: la domanda non è posta solo a chi ha avuto l'infarto. La quota di risposta è quasi uguale nei due gruppi (`CVDASPRN`: 4,53% fra i positivi e 4,25% fra i negativi; `DRADVISE`: 11,4% e 9,1%). Chi ha fatto il modulo ha risposto, indipendentemente dal target;
- **è però un proxy post-outcome fortissimo**: chi prende aspirina ha 4,6 volte la prevalenza di infarto di chi non la prende (20,8% vs 4,5%). Il legame è vero, ma è "l'infarto → il medico prescrive l'aspirina", non il contrario.

**Decisione**: le teniamo **fuori dal dataset base** e le testiamo in cross-validation come *ablation*. Se la CV dice che guadagnano, le riaggiungiamo al modello finale e lo dichiariamo apertamente nel report (sono variabili "dopo l'evento", non cause). Se la CV dice che non guadagnano, avremo la prova che il sospetto era giusto. Toglierle a mano, senza misurare, sarebbe un'affermazione non supportata dai dati.

In [58]:
# --- Evidenze numeriche per la scelta dei 3 candidati ---
# (stesse regole di informativita_colonne: quota di risposta, spread, tasso per valore)
y01 = (y_train == 1).astype(float)
nan_col = np.isnan(x_train).mean(axis=0)
ix = {c: i for i, c in enumerate(cols)}

print("%-10s %7s %8s %9s %10s %10s" % ("colonna", "%NaN", "quota", "spread", "infarto sì", "infarto no"))
for nome in CANDIDATI:
    j = ix[nome]
    col = x_train[:, j]
    quota = 1 - nan_col[j]
    risp = ~np.isnan(col)
    valori = np.unique(col[risp])
    tassi = []
    for v in valori:
        m = risp & (col == v)
        tassi.append(y01[m].mean())
    spread = 100 * np.sum((np.bincount(np.unique(col[risp], return_inverse=True)[1], minlength=len(valori))
                            / risp.sum()) * np.abs(np.array(tassi) - y01[risp].mean()))
    s = " / ".join("%.1f%%" % (100 * t) for t in tassi)
    print("%-10s %6.1f%% %7.1f%% %8.2f %10s" % (nome, 100 * nan_col[j], 100 * quota, spread, s))

# la domanda e' posta a tutti quelli che hanno fatto il modulo?
print("\nQuota di risposta fra positivi e negativi (se fosse leakage strutturale, i positivi risponderebbero di piu'):")
for nome in CANDIDATI:
    j = ix[nome]
    risp = ~np.isnan(x_train[:, j])
    print("   %-10s positivi %5.2f%%   negativi %5.2f%%" % (
        nome, 100 * risp[y_train == 1].mean(), 100 * risp[y_train == -1].mean()))
print("(quota di risposta simile nei due gruppi = la domanda non e' condizionata al target)")

colonna       %NaN    quota    spread infarto sì infarto no
INSULIN      93.4%     6.6%     3.98 28.0% / 19.0%
CVDASPRN     95.7%     4.3%     6.86 20.8% / 4.5%
DRADVISE     90.7%     9.3%     6.03 20.7% / 6.5%

Quota di risposta fra positivi e negativi (se fosse leakage strutturale, i positivi risponderebbero di piu'):
   INSULIN    positivi 16.41%   negativi  5.66%
   CVDASPRN   positivi  4.53%   negativi  4.25%
   DRADVISE   positivi 11.37%   negativi  9.06%
(quota di risposta simile nei due gruppi = la domanda non e' condizionata al target)


### Come ricostruiamo `INSULIN`

Il NaN di `INSULIN` non significa una cosa sola, quindi la colonna grezza va sostituita da una **categorica con 5 livelli** costruita con `DIABETE3` (che resta nel dataset: 0,1% di NaN) e `INSULIN`:

| valore | significato | n | tasso di infarto |
|---|---|---|---|
| 0 | non diabetico (`DIABETE3` ≠ 1) | 285.433 | 6,86% |
| 1 | diabetico, ma il modulo non è stato somministrato | 20.535 | 22,35% |
| 2 | ignoto (`DIABETE3` mancante) | 479 | 12,32% |
| 3 | diabetico **+ insulina** | 7.133 | **27,98%** |
| 4 | diabetico, **senza insulina** | 14.555 | 18,96% |

Le categorie 1 e 3-4 sono quella che "paga": il modello vede che il diabete è di per sé un fattore di rischio (22% vs 7%) e che l'insulina aggiunge altro (28% vs 19%).

⚠️ **Da chiarire sul codebook**: `DIABETE3` ha **4 valori**, non 3 — `1` = diabetico (42.223, 22,1% di infarti), `3` = non diabetico (277.049, 6,7%), `4` (5.709, **14,5%**) e `2` (2.675, 4,0%). I valori 2 e 4 sono categorie extra (borderline / prediabete?): il 14,5% del valore 4 **non va fuso** con il "non diabetico" al 6,9%. La `cleaning.py` attuale già assume `DIABETE3 ∈ {2,3,4}` = non diabetici (per mettere 0 in `DIABAGE2`), quindi è coerente, ma la separazione 2/4 va verificata.

**Nota sul train/test**: le regole sono solo basate sui *valori* (non su statistiche aggregate), quindi applico **esattamente le stesse regole** a train e test.

In [59]:
def categorizza_insulin(dia, ins):
    """
    dia : valori di DIABETE3 (NaN = ignoto)
    ins : valori di INSULIN   (NaN = modulo non risposto)
    Ritorna le 5 categorie descritte nella cella markdown precedente.
    """
    cat = np.zeros(len(dia), dtype=int)          # 0 = non diabetico
    cat[np.isnan(dia)] = 2                       # 2 = ignoto
    cat[np.isin(dia, [1])] = 1                   # 1 = diabetico (modulo non fatto)
    cat[np.isin(dia, [1]) & (ins == 1)] = 3      # 3 = diabetico + insulina
    cat[np.isin(dia, [1]) & (ins == 2)] = 4      # 4 = diabetico senza insulina
    return cat

# 1) costruiamo la categoria PRIMA di buttare via la colonna grezza
ix = {c: i for i, c in enumerate(cols)}
assert "DIABETE3" in ix and "INSULIN" in ix
ins_cat_tr = categorizza_insulin(x_train[:, ix["DIABETE3"]], x_train[:, ix["INSULIN"]])
ins_cat_te = categorizza_insulin(x_test[:,  ix["DIABETE3"]], x_test[:,  ix["INSULIN"]])

# controllo: quante persone in ogni categoria (train)
nomi = {0: "non diabetico", 1: "diabete, modulo non fatto", 2: "ignoto",
        3: "diabete + insulina", 4: "diabete, senza insulina"}
y01 = (y_train == 1).astype(float)
print("INSULIN ricostruito in 5 categorie (train):")
for k in sorted(nomi):
    m = ins_cat_tr == k
    print("   %d  %-26s n=%6d   tasso infarti %5.2f%%" % (k, nomi[k], m.sum(), 100 * y01[m].mean()))

# 2) ora tagliamo: le 120 colonne da buttare (123 - i 3 candidati)
x_train, cols_new = drop_columns(x_train, cols, drop_finale)
x_test,  cols_te  = drop_columns(x_test,  cols, drop_finale)
assert cols_new == cols_te, "train e test non piu' allineati"
cols = cols_new
print("\n--> tolte %d colonne: restano %d" % (len(drop_finale), len(cols)))

# 3) INSULIN grezza -> INSULIN_CAT (INSULIN era tra le 123, quindi non c'era piu')
x_train, cols_new = drop_columns(x_train, cols, ["INSULIN"])
x_test,  cols_te  = drop_columns(x_test,  cols, ["INSULIN"])
assert cols_new == cols_te
cols = cols_new

x_train = np.column_stack([x_train, ins_cat_tr])
x_test  = np.column_stack([x_test,  ins_cat_te])
cols = cols + ["INSULIN_CAT"]
print("--> aggiunta la colonna INSULIN_CAT, shape:", x_train.shape, x_test.shape)

INSULIN ricostruito in 5 categorie (train):
   0  non diabetico              n=285433   tasso infarti  6.86%
   1  diabete, modulo non fatto  n= 20535   tasso infarti 22.35%
   2  ignoto                     n=   479   tasso infarti 12.32%
   3  diabete + insulina         n=  7133   tasso infarti 27.98%
   4  diabete, senza insulina    n= 14555   tasso infarti 18.96%

--> tolte 120 colonne: restano 153
--> aggiunta la colonna INSULIN_CAT, shape: (328135, 153) (109379, 153)


### Dove siamo arrivati

- **321** colonne originali → **273** dopo il taglio "a priori" → **153** dopo il taglio automatico del 60% di NaN e la sostituzione di `INSULIN` con `INSULIN_CAT`.
- I NaN rimasti sono **NaN veri**: interviste interrotte, "non so / rifiuta", moduli opzionali non somministrati → il prossimo passo è l'imputazione.
- Restano **153** colonne (di cui 141 con almeno un NaN), con l'unica eccezione di `CVDASPRN` e `DRADVISE`, che hanno ancora > 60% di NaN perché **le teniamo dentro apposta**: sono le 2 candidate "contese" (vedi la cella finale, che costruisce anche la versione *base* senza di loro).

**Appunti per il report (ablation):**
1. il criterio "> 60% di NaN" è fragile su `BPMEDS` (59,9%: tenuta per 0,1 punti percentuali) e incoerente su `SMOKDAY2` (58,4%, tenuta) vs `LASTSMK2` (72,5%, tolta) → da confrontare in CV con soglie 0,55 / 0,65;
2. fra le 153 restano **coppie ridondanti** calcolate dal CDC (`_BMI5`/`_BMI5CAT`/`_RFBMI5`, `TOLDHI2`/`_RFCHOL`/`CHOLCHK`/`_CHOLCHK`, `GENHLTH`/`_RFHLTH`, `SEATBELT`/`_RFSEAT2`/`_RFSEAT3`, `INCOME2`/`_INCOMG`) → secondo giro di potatura, utile per l'ablation;
3. `_HCVU651` (35,9% di NaN): il 9 significa "over 65, non applicabile" → va trattato come **categoria**, non come NaN da imputare;
4. `CVDASPRN` e `DRADVISE` sono **proxy post-outcome** (l'aspirina viene prescritta *dopo* l'infarto): non è leakage strutturale, ma il legame è fortissimo. Le teniamo in `CANDIDATI` e le confrontiamo in CV: il modello *base* le esclude, l'*ablazione* le aggiunge.

In [60]:
# --- Riepilogo finale ---
print("=" * 70)
print("SHAPE train:", x_train.shape, " test:", x_test.shape)
print("colonne:", len(cols))
print("NaN residui: train %.1f%%   test %.1f%%" % (100 * np.isnan(x_train).mean(), 100 * np.isnan(x_test).mean()))
print("colonne con > 60% di NaN rimaste:", [c for c in cols if np.isnan(x_train[:, cols.index(c)]).mean() > 0.6])
print("colonne con > 0 NaN:", int((np.isnan(x_train).mean(axis=0) > 0).sum()))
print("-" * 70)
print("candidati 'contese' (restano nel dataset ma vanno testate in CV):")
for c in CANDIDATI:
    print("   %-10s presente nel dataset finale: %s" % (c, c in cols))

# --- la matrice BASE per il primo modello: senza le 2 candidate (l'aspirina) ---
#     INSULIN non c'e' piu': e' stata sostituita da INSULIN_CAT
CANDIDATI_ANCORA_NEL_DATASET = [c for c in CANDIDATI if c in cols]
x_base, cols_base = drop_columns(x_train, cols, CANDIDATI_ANCORA_NEL_DATASET)
x_base_test, cols_base_test = drop_columns(x_test, cols, CANDIDATI_ANCORA_NEL_DATASET)
assert cols_base == cols_base_test
print("-" * 70)
print("matrice BASE (senza %s): %s" % (CANDIDATI_ANCORA_NEL_DATASET, x_base.shape))
print("matrix FULL (con le candidate): %s" % (x_train.shape,))
print("-" * 70)
print("prime 20 colonne finali:", cols[:20])
print("ultime 10 colonne finali:", cols[-10:])
print("=" * 70)
print("PROSSIMO PASSO: preprocessing (imputazione mediana/moda calcolata SOLO sul train,")
print("               indicatori di missing, one-hot delle categoriche, standardizzazione)")

SHAPE train: (328135, 153)  test: (109379, 153)
colonne: 153
NaN residui: train 8.3%   test 8.3%
colonne con > 60% di NaN rimaste: ['DRADVISE', 'CVDASPRN']
colonne con > 0 NaN: 141
----------------------------------------------------------------------
candidati 'contese' (restano nel dataset ma vanno testate in CV):
   INSULIN    presente nel dataset finale: False
   CVDASPRN   presente nel dataset finale: True
   DRADVISE   presente nel dataset finale: True
----------------------------------------------------------------------
matrice BASE (senza ['CVDASPRN', 'DRADVISE']): (328135, 151)
matrix FULL (con le candidate): (328135, 153)
----------------------------------------------------------------------
prime 20 colonne finali: ['GENHLTH', 'PHYSHLTH', 'MENTHLTH', 'POORHLTH', 'HLTHPLN1', 'PERSDOC2', 'MEDCOST', 'CHECKUP1', 'BPHIGH4', 'BPMEDS', 'BLOODCHO', 'CHOLCHK', 'TOLDHI2', 'CVDSTRK3', 'ASTHMA3', 'CHCSCNCR', 'CHCOCNCR', 'CHCCOPD1', 'HAVARTH3', 'ADDEPEV2']
ultime 10 colonne finali: ['_P

## Spiegazione delle colonne ridondanti

Dopo il taglio del 60% di NaN restano **153 colonne**, ma molte contengono **due volte la stessa informazione**. Non lo abbiamo dedotto a occhio: l'abbiamo **misurato sui dati**.

**Come si fa.** Per ogni coppia di colonne chiediamo se una sia una *funzione deterministica* dell'altra: fissato il valore di una, l'altra ha **sempre** lo stesso valore? Se sì, la seconda non aggiunge niente. Chiamiamo *purezza* la frazione di righe in cui la relazione vale (il caso "manca" è contato come un valore a sé, quindi confrontiamo anche il *pattern* dei NaN). Con purezza = 1,0000 la colonna è **letteralmente** ridondante.

Sulle 153 colonne abbiamo trovato **30 coppie**, tutte con purezza 1,0000.

### Le dividiamo in tre gruppi, da trattare in modo diverso

**1) Copie (24 colonne) → le togliamo subito.** La figlia è deterministica dalla madre: buttarla **non perde informazione**.

| famiglia | tengo | butto |
|---|---|---|
| salute generale | `GENHLTH` | `_RFHLTH` |
| ipertensione | `BPHIGH4` (+ `BPMEDS`) | `_RFHYPE5` |
| colesterolo | `TOLDHI2` | `_RFCHOL` |
| asma | `ASTHMA3`, **`_ASTHMS1`** | `_LTASTH1`, `_CASTHM1` |
| artrite | `HAVARTH3` | `_DRDXAR1` |
| fumo | `_SMOKER3`, `SMOKE100` | `SMOKDAY2`, `_RFSMOK3` |
| attività fisica | `EXERANY2`, `_PACAT1` | `_TOTINDA`, `_PA30021`, `_PAINDX1`, `_PASTAE1` |
| età | `_AGE80`, `_AGEG5YR` | `_AGE_G` |
| razza | `_RACE` | `_RACEG21`, `_RACEGR3`, `_RACE_G1` |
| reddito | `INCOME2` | `_INCOMG` |
| istruzione | `EDUCA` | `_EDUCAG` |
| cintura di sicurezza | `SEATBELT` | `_RFSEAT2`, `_RFSEAT3` |
| test HIV | `HIVTST6` | `_AIDTST3` |
| altezza | `HTM4` | `HTIN4` (pollici ↔ cm) |
| capacità funzionale | **`MAXVO2_`** | `FC60_` |
| BMI | `_BMI5CAT` | `_RFBMI5` |

> **`_ASTHMS1` e `MAXVO2_` le teniamo** (2 colonne in più, nessun impatto sulla metrica): il
> test automatico non trovava per loro una "madre" singola, quindi per prudenza restano nel
> dataset. `_ASTHMS1` distingue "asma ancora attuale" da "asma in passato" (cosa che `ASTHMA3`
> non dice) e `MAXVO2_` (capacità funzionale) non è determinata da età/sex/BMI.

**2) Quasi costanti (11 colonne) → aspettiamo la CV.** Queste non sono ridondanti: sono colonne che *non variano*. Lo *spread* (quanto cambia il tasso di infarto fra i valori della colonna) è sotto 0,3 punti percentuali, e per quattro di loro la differenza riguarda 39-70 righe su 328.135. Non le buttiamo ancora: la decisione la prende la CV, e intanto le teniamo in lista.

**3) Non sono ridondanti, solo parentele → non si toccano.** `CHOLCHK` (hai fatto il test?), `BLOODCHO` (quanto?), `TOLDHI2` (ti hanno detto che è alto?), `_CHOLCHK` (ricodifica del CDC) sono **quattro domande diverse**. Idem `SMOKE100` vs `_SMOKER3`, `PHYSHLTH`/`MENTHLTH`/`POORHLTH`, `DIABETE3`/`DIABAGE2`, `_RACE` vs `_PRACE1`/`_HISPANC`.

### Perché togliere le copie (motivi tecnici, non estetici)

- **Ridge**: due colonne identiche si *dividono* la penalità L2 → la regolarizzazione effettiva su quella direzione si indebolisce → più overfitting. L'effetto è reale e misurabile.
- **One-hot**: `_RACE` + `_RACEG21` + `_RACEGR3` + `_RACE_G1` diventano 4 blocchi di variabili dummy identici → collinearità e coefficienti instabili (male per il report).
- **Dimensione**: 398 → ~336 feature dopo il taglio delle copie.
- **Leggibilità**: 153 → 129 colonne, molto più difendibili davanti a chi legge.

### La coppia "difficile" (e perché NON la decidiamo a mano)

`_BMI5` (continuo) e `_BMI5CAT` (4 classi) sono l'uno la discretizzazione dell'altro: sembrerebbe una copia. Ma il rischio di infarto in funzione del BMI ha una **forma a U**, e per un modello *lineare* la versione categorizzata può essere **migliore** di quella continua. Stesso discorso per l'età (`_AGE80` continuo, `_AGEG5YR` a 13 classi, `_AGE65YR` binaria).

Per queste la ridondanza è solo **apparente**: restano tutte e tre, e la scelta la farà la **cross-validation** (è la riga "grana diverse" della tabella di ablation).

In [61]:
# ===================================================================
# Colonne ridondanti: rilevazione automatica + controlli
# ===================================================================
import importlib, cleaning
importlib.reload(cleaning)                       # per prendere le modifiche a cleaning.py
from cleaning import colonne_ridondanti, drop_columns

ridondanti = colonne_ridondanti(x_train, cols)
print("coppie ridondanti trovate:", len(ridondanti))
for m, f, p in ridondanti:
    print(f"   {f:12s} e' funzione di {m:12s}  (purezza {p:.4f})")

# --- le due liste di decisione (le usiamo anche per l'ablation in CV) ---
DA_TOGLIERE_COPIE = ['_RFHLTH', '_RFHYPE5', '_RFCHOL', '_LTASTH1', '_CASTHM1', '_DRDXAR1',
                     '_EDUCAG', '_INCOMG', '_RFSMOK3', 'SMOKDAY2', '_RFSEAT2', '_RFSEAT3',
                     '_AIDTST3', '_RACEG21', '_RACEGR3', '_RACE_G1', '_AGE_G', '_TOTINDA',
                     '_PA30021', '_PAINDX1', '_PASTAE1', 'HTIN4', '_RFBMI5', 'FC60_']
DA_TOGLIERE_VUOTE = ['_FRT16', '_VEG23', '_FRUITEX', '_VEGETEX', '_FRTRESP', '_VEGRESP',
                     '_MISFRTN', '_MISVEGN', 'USENOW3', 'PAMISS1_', '_HCVU651']

# --- CONTROLLO 1: ogni copia buttata deve essere determinata da UNA colonna che
#     resta -> buttarla non perde informazione.
#     Verifica DIRETTA (il detector qui sopra lavora solo su coppie categoriche
#     con >= 500 righe per classe, quindi non vede _EDUCAG, _AGE_G e HTIN4).
def madri_che_determinano(X, cols, figlie, madri_idx):
    """Per ogni figlia restituisce il nome di una madre superstite che la determina (None se none)."""
    codes = np.column_stack([np.unique(X[:, j], return_inverse=True)[1] for j in figlie])
    esito = {j: None for j in figlie}
    for k in madri_idx:
        if all(esito[j] is not None for j in figlie):
            break
        _, g = np.unique(X[:, k], return_inverse=True)
        order = np.argsort(g, kind="stable")
        gs = g[order]
        dentro = gs[1:] == gs[:-1]
        for t, j in enumerate(figlie):
            if esito[j] is not None:
                continue
            vs = codes[order, t]
            if not np.any(dentro & (vs[1:] != vs[:-1])):
                esito[j] = cols[k]
    return esito

def madre_che_determina_i_valori(X, cols, figlia_idx, madri_idx):
    """Come sopra, ma ignora i NaN della figlia: serve per il caso 'stessa
    grandezza in unita' diverse' (i valori coincidono, i NaN no)."""
    v = X[:, figlia_idx]
    idx = np.where(~np.isnan(v))[0]
    cod = np.unique(v[idx], return_inverse=True)[1]
    for k in madri_idx:
        g = np.unique(X[:, k], return_inverse=True)[1]
        gg = g[idx]
        order = np.argsort(gg, kind="stable")
        gs = gg[order]
        vs = cod[order]
        dentro = gs[1:] == gs[:-1]
        if not np.any(dentro & (vs[1:] != vs[:-1])):
            return cols[k]
    return None

figlie_idx = [cols.index(c) for c in DA_TOGLIERE_COPIE]
madri_idx = [i for i, c in enumerate(cols) if c not in DA_TOGLIERE_COPIE]
print("\nCONTROLLO 1 - madri che restano per ogni copia buttata:")
esito = madri_che_determinano(x_train, cols, figlie_idx, madri_idx)
for c, j in zip(DA_TOGLIERE_COPIE, figlie_idx):
    m = esito[j]
    if m:
        print(f"   {c:12s} -> resta {m}")
    else:
        # nessuna madre esatta: vediamo se e' la stessa grandezza in unita' diverse
        m2 = madre_che_determina_i_valori(x_train, cols, j, madri_idx)
        if m2:
            k = cols.index(m2)
            nn = np.isnan(x_train[:, j]) != np.isnan(x_train[:, k])
            print(f"   {c:12s} -> nessuna madre ESATTA, ma i valori coincidono con {m2} "
                  f"(stessa grandezza in unita' diverse); i NaN non coincidono su "
                  f"{int(nn.sum())} righe ({100*nn.mean():.2f}%) -> nessuna perdita")
        else:
            print(f"   {c:12s} -> !!! NESSUNA MADRE (controllare)")

# --- CONTROLLO 2: le colonne "vuote" variano davvero? (spread = info) ---
def spread_colonna(nome, X, y):
    """Quanto cambia il tasso di infarti fra i valori della colonna."""
    j = cols.index(nome)
    v = X[:, j]
    m = ~np.isnan(v)
    base = y[m].mean()
    u = np.unique(v[m])
    if len(u) > 20:                      # continua: spezzata in 10 gruppi
        q = np.unique(np.quantile(v[m], np.linspace(0, 1, 11)[1:-1]))
        g = np.searchsorted(q, v[m], side="right")
    else:
        g = np.searchsorted(u, v[m])
    ng = np.bincount(g)
    pg = np.bincount(g, weights=y[m].astype(float))
    ok = ng > 0
    return 100 * np.sum(ng[ok] / ng.sum() * np.abs(pg[ok] / ng[ok] - base))

y01 = (y_train == 1).astype(float)
print("\nCONTROLLO 2 - le colonne 'vuote' (informazione trascurabile):")
print(f"   {'colonna':12s}{'spread':>8}{'classi':>8}{'min classe':>12}{'%NaN':>8}")
for c in DA_TOGLIERE_VUOTE:
    j = cols.index(c)
    v = x_train[:, j]
    m = ~np.isnan(v)
    u, cnt = np.unique(v[m], return_counts=True)
    print(f"   {c:12s}{spread_colonna(c, x_train, y01):8.2f}{len(u):8d}{cnt.min():12d}"
          f"{100*np.isnan(v).mean():8.1f}")

coppie ridondanti trovate: 30
   _RFHLTH      e' funzione di GENHLTH       (purezza 1.0000)
   _RFHYPE5     e' funzione di BPHIGH4       (purezza 1.0000)
   _RFCHOL      e' funzione di TOLDHI2       (purezza 1.0000)
   _LTASTH1     e' funzione di ASTHMA3       (purezza 1.0000)
   _DRDXAR1     e' funzione di HAVARTH3      (purezza 1.0000)
   _INCOMG      e' funzione di INCOME2       (purezza 1.0000)
   SMOKDAY2     e' funzione di _SMOKER3      (purezza 1.0000)
   _TOTINDA     e' funzione di EXERANY2      (purezza 1.0000)
   _RFSEAT2     e' funzione di SEATBELT      (purezza 1.0000)
   _RFSEAT3     e' funzione di SEATBELT      (purezza 1.0000)
   _AIDTST3     e' funzione di HIVTST6       (purezza 1.0000)
   _CASTHM1     e' funzione di _ASTHMS1      (purezza 1.0000)
   _RACEG21     e' funzione di _RACE         (purezza 1.0000)
   _RACEGR3     e' funzione di _RACE         (purezza 1.0000)
   _RACE_G1     e' funzione di _RACE         (purezza 1.0000)
   _RACEG21     e' funzione di _RACEGR3 

### Applicazione del taglio

**Ora applichiamo solo le 24 copie.** Sono l'unico gruppo la cui rimozione è *provata* senza perdita di informazione: la cella precedente verifica, per ciascuna, che esista una colonna **superstite** che la determina.

**Cosa NON applichiamo ancora, e perché:**

- le **11 colonne quasi costanti**: lo spread basso è un forte indizio che non servano, ma la conferma la diamo con la CV (e ci serve una riga nella tabella di ablation);
- le **coppie "grana diverse"** (`_BMI5`/`_BMI5CAT`, `_AGE80`/`_AGEG5YR`/`_AGE65YR`): la versione categorizzata può essere migliore per un modello lineare → decide la CV;
- **`_ASTHMS1` e `MAXVO2_`**: si **tengono** (il test automatico non trovava per loro una "madre" singola, quindi per prudenza restano nel dataset).

⚠️ **Un ordine da rispettare**: `BPMEDS` è al **59,9%** di NaN, cioè *sulla soglia* del 60%. Se in futuro decidessimo di buttare anche `BPMEDS`, allora `_RFHYPE5` (il flag binario "ipertensione") diventerebbe l'unica variabile d'ipertensione rimasta, e andrebbe recuperata. Per ora `BPMEDS` resta, quindi `_RFHYPE5` può essere buttata.

Le due liste restano nel notebook: l'*ablation* "con / senza colonne ridondanti" è una riga di codice, e nel report ci va la riga **"−24 colonne ridondanti: score CV da X a Y"**.

In [62]:
# ===================================================================
# Applico il taglio delle 24 copie (train e test insieme)
# ===================================================================
print("prima del taglio:  train", x_train.shape, " test", x_test.shape)

x_train, cols_new = drop_columns(x_train, cols, DA_TOGLIERE_COPIE)
x_test,  cols_te  = drop_columns(x_test,  cols, DA_TOGLIERE_COPIE)
assert cols_new == cols_te, "train e test non piu' allineati!"
cols = cols_new

print("dopo  il taglio:  train", x_train.shape, " test", x_test.shape)
print("colonne tolte:", len(DA_TOGLIERE_COPIE))
print("NaN residui: train %.1f%%   test %.1f%%" % (100 * np.isnan(x_train).mean(),
                                                   100 * np.isnan(x_test).mean()))
print("colonne con > 60% di NaN rimaste:", [c for c in cols
                                            if np.isnan(x_train[:, cols.index(c)]).mean() > 0.6])
print("colonne tenute per prudenza: _ASTHMS1 (asma attuale vs passato), MAXVO2_ (capacita' funzionale)")

# --- ricostruisco anche la matrice BASE (senza le 2 candidate 'contese') ---
CANDIDATI_ANCORA_NEL_DATASET = [c for c in CANDIDATI if c in cols]
x_base, cols_base = drop_columns(x_train, cols, CANDIDATI_ANCORA_NEL_DATASET)
x_base_test, cols_base_test = drop_columns(x_test, cols, CANDIDATI_ANCORA_NEL_DATASET)
assert cols_base == cols_base_test
print("matrice BASE (senza %s): %s" % (CANDIDATI_ANCORA_NEL_DATASET, x_base.shape))

# --- le 11 'vuote' restano in lista: le applichiamo dopo la CV ---
#     per usarle basta una riga:
#     x_train, cols = drop_columns(x_train, cols, DA_TOGLIERE_VUOTE)
print("\nDA_TOGLIERE_VUOTE (in attesa della CV):", len(DA_TOGLIERE_VUOTE), DA_TOGLIERE_VUOTE)
print("colonne finali:", len(cols))
print("=" * 70)

prima del taglio:  train (328135, 153)  test (109379, 153)
dopo  il taglio:  train (328135, 127)  test (109379, 127)
colonne tolte: 26
NaN residui: train 8.4%   test 8.4%
colonne con > 60% di NaN rimaste: ['DRADVISE', 'CVDASPRN']
matrice BASE (senza ['CVDASPRN', 'DRADVISE']): (328135, 125)

DA_TOGLIERE_VUOTE (in attesa della CV): 11 ['_FRT16', '_VEG23', '_FRUITEX', '_VEGETEX', '_FRTRESP', '_VEGRESP', '_MISFRTN', '_MISVEGN', 'USENOW3', 'PAMISS1_', '_HCVU651']
colonne finali: 127
